## Capítulo 1: Simulação de Redes Hidráulicas
---

Este relatório cobre a modelagem e simulação de redes de microcanais hidráulicos utilizando o método dos elementos de rede (grafos).No caso de sistemas microfluídicos contendo fluidos Newtonianos.


Os fundamentos teóricos da modelagem hidráulica baseiam-se na lei constitutiva para escoamento laminar em tubos circulares, que relaciona a vazão volumétrica $Q$ à diferença de pressão $\Delta P$ por meio da condutância 
$ C_k = \frac{\pi D_h^4}{128\, \mu\, L}, $ 
onde $D_h = \sqrt{4A/\pi}$ é o diâmetro hidráulico calculado a partir da área da seção transversal $A$ e $\mu$ é a viscosidade dinâmica do fluido. As condições de contorno são aplicadas de duas formas: nos nós com pressão conhecida, substitui-se a linha correspondente do sistema por $1 \cdot p_i = p_{\text{ref}}$; nos nós com vazão imposta, o valor $Q_{\text{in}}$ é inserido diretamente no vetor de fontes $b_i$. Após a obtenção do campo de pressões, as vazões em cada canal são determinadas por 
$\mathbf{Q} = \mathbf{K}\, \mathbf{D}\, \mathbf{p},$ 
e a potência dissipada na rede é calculada como 
$W = \mathbf{p}^T \mathbf{D}^T \mathbf{Q},$
em que $\mathbf{K}$ é a matriz diagonal das condutâncias dos canais e $\mathbf{D}$ é a matriz de incidência nó-aresta do grafo.

In [ ]:
# Importações!!!
import os
import sys
import importlib
import matplotlib.pyplot as plt

root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if root not in sys.path:
    sys.path.insert(0, root)

import env
import hydraulics
import analysis
import data_structures

importlib.reload(env)
importlib.reload(hydraulics)
importlib.reload(analysis)
importlib.reload(data_structures)

config = env.CONFIG_H


Os parâmetros de configuração da simulação estão centralizados no dicionário CONFIG_H, definido no arquivo env.py, o que permite alterar as condições de contorno sem modificar as classes de simulação. Entre os principais parâmetros, destacam-se: o índice do nó de entrada (N_INLET = 0), a vazão de entrada padrão (INLET_FLOW = 1×10⁻⁷ m³/s), o índice do nó de saída (N_OUTLET = 5), cuja pressão de referência é fixada em OUTLET = 0 Pa, a área da seção transversal dos canais (PIPE_AREA = 2,5×10⁻⁷ m²) e a viscosidade dinâmica do fluido (VISCOSITY = 0,001 Pa·s), valor típico da água a aproximadamente 20 °C.

---
## Geração da Topologia da Rede (`GeraGrafo`)

In [ ]:
# ---- Geração e visualização da rede (levels=3) ----
Xno, conec = data_structures.GeraGrafo(levels=3)
Xno_m = Xno * 0.001   # converte de mm para metros

print(f'Rede gerada:')
print(f'  Nós   : {Xno.shape[0]}')
print(f'  Canais: {conec.shape[0]}')

fig, ax = plt.subplots(figsize=(14, 5))
for (i, j) in conec:
    ax.plot([Xno[i, 0], Xno[j, 0]], [Xno[i, 1], Xno[j, 1]],
            'k-', linewidth=0.7, zorder=1)
ax.scatter(Xno[:, 0], Xno[:, 1], s=10, color='royalblue', zorder=2)
ax.scatter(Xno[config['N_INLET'], 0], Xno[config['N_INLET'], 1],
           s=120, color='green', zorder=3, label=f'Inlet (nó {config["N_INLET"]})')
ax.scatter(Xno[config['N_OUTLET'], 0], Xno[config['N_OUTLET'], 1],
           s=120, color='red', zorder=3, label=f'Outlet (nó {config["N_OUTLET"]})')
ax.set_aspect('equal'); ax.axis('off')
ax.set_title('Topologia Fractal da Rede de Microcanais (levels=3)', fontsize=13)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

---
## 5. Classe Base `Hydraulics`

A classe encapsula os três passos fundamentais do método:

```
calculate_conductancy()   ──►   assembly()   ──►   solve_network()   ──►   calculate_flow_rate_and_potency()
     C_k (por canal)/pressões p/vazões Q, potência W
```
### 5.1 Cálculo das Condutâncias

Para cada canal $k$ conectando nós $i$ e $j$:

$C_k = \frac{\pi D_h^4}{128\, \mu\, L_k}, \qquad L_k = \|\mathbf{x}_i - \mathbf{x}_j\|$
### 5.2 Aplicação das Condições de Contorno e Solução

Linha do nó `outlet` é substituída por $[0 \ldots 1 \ldots 0]\,\mathbf{p} = p_{\text{ref}}$;  
o sistema resultante é resolvido por `numpy.linalg.solve` (eliminação LU densa).

In [ ]:
solver_base = hydraulics.Hydraulics(conec, Xno_m, config)
solver_base.run(print_info=True, plot=True)

O Problema 1 trata de múltiplas entradas de vazão,A modificação consiste em alterar o vetor de fontes `b` para receber várias vazões simultâneas, cada uma indexada a um nó diferente, de acordo com `b_i = Q_in,i` para todo `i` pertencente ao dicionário de vazões de entrada. A montagem da matriz `Ã` permanece idêntica à do caso base, mudando-se unicamente a construção do vetor de fontes.

In [ ]:
solver_p1 = hydraulics.Hydraulics_p1(conec, Xno_m, config)
solver_p1.run(print_info=True, plot=True)

O Problema 2 trata de múltiplas entradas de pressão, onde a principal modificação consiste em aplicar condições de Dirichlet nos nós com pressão prescrita. Para cada um desses nós, a linha correspondente na matriz `Ã` é substituída por uma equação que impõe diretamente o valor da pressão: o elemento diagonal `A_ii` recebe 1, todos os elementos fora da diagonal (`A_ij` para `j ≠ i`) são zerados, e o termo independente `b_i` recebe a pressão `p_i` especificada.

In [ ]:
solver_p2 = hydraulics.Hydraulics_p2(conec, Xno_m, config)
solver_p2.run(print_info=True, plot=True)

O Problema 3 aborda a imposição de pressão na entrada, com o objetivo de calcular a vazão resultante de uma diferença de pressão fixada entre o inlet e o outlet. Para isso, tanto a entrada quanto a saída recebem condições de contorno de Dirichlet para a pressão, de modo que o sistema linear resolve diretamente o campo de pressões. A vazão de entrada `Q_in` é então recuperada posteriormente, utilizando a linha original da matriz,antes de ser zerada para aplicação da condição de contorno,por meio do produto interno `Q_in = a_inlet^T · p`.

In [ ]:
solver_p3 = hydraulics.Hydraulics_p3(conec, Xno_m, config)
solver_p3.run(print_info=True, plot=True)

O Problema 4 trata de uma vazão senoidal no tempo, descrita por `Q(t) = A sin(ω t + θ) + B`, onde uma componente oscilatória senoidal se sobrepõe a um valor constante. A estratégia de solução explora a superposição linear, já que o sistema `Ã p = Q` é linear e invariante no tempo. Dessa forma, é necessário resolver o sistema apenas duas vezes: uma para obter o campo de pressões `p_sin` correspondente a uma amplitude unitária senoidal (`Q = 1` mL/s) e outra para obter `p_const` associada ao termo constante `Q = B` mL/s. A pressão em qualquer instante `t` é então calculada diretamente pela combinação `p(t) = A sin(ω t + θ) · p_sin + p_const`, evitando a resolução do sistema linear a cada passo de tempo e proporcionando um grande ganho de eficiência computacional.

In [ ]:
solver_p4 = hydraulics.Hydraulics_p4(conec, Xno_m, config)
solver_p4.run(print_info=True, plot=True)

O Problema 5 aborda a superposição de excitações seno e cosseno, há presença de dois inlets independentes com comportamentos oscilatórios distintos. No nó 0, a vazão é descrita por `Q1(t) = A1 sin(ω1 t + θ1) + B1`, enquanto no nó 175 a vazão segue `Q2(t) = A2 cos(ω2 t + θ2) + B2`. A estratégia de solução baseia-se na linearidade do sistema, utilizando quatro soluções-base: resposta ao seno de amplitude unitária, à constante associada ao seno, ao cosseno de amplitude unitária e à constante associada ao cosseno. Essas soluções são pré-calculadas uma única vez e, para cada instante `t`, o campo de pressões é obtido pela combinação linear `p(t) = f_sin(t)·p_sin + p_sin,const + f_cos(t)·p_cos + p_cos,const`, em que os fatores `f_sin(t)` e `f_cos(t)` carregam as amplitudes e fases instantâneas das excitações, evitando a necessidade de resolver o sistema linear a cada passo de tempo.

In [ ]:
solver_p5 = hydraulics.Hydraulics_p5(conec, Xno_m, config)
solver_p5.run(print_info=True, plot=True)

O Problema 6 aborda a viscosidade dependente da temperatura, motivado pelo fato da viscosidade do fluido ser alterada pela temperatura. A temperatura evolui com o tempo segundo `T(t) = 20 + 0.9 t²` [°C], e a viscosidade da água é descrita pela correlação empírica `μ(T) = 0.001791 / (1 + 0.03368 T + 0.000221 T²)` [Pa·s]. A estratégia de solução baseia-se na linearidade: como as condutâncias são proporcionais a `1/μ` e a matriz do sistema é proporcional às condutâncias, as pressões escalam linearmente com `μ`. Dessa forma, a pressão no instante `t` pode ser obtida por `p(t) = (μ(T(t))/μ0) · p0`, exigindo apenas uma única resolução do sistema linear para uma viscosidade de referência `μ0` e um simples rescalamento para cada instante subsequente, sem necessidade de resolver novos sistemas.

In [ ]:
solver_p6 = hydraulics.Hydraulics_p6(conec, Xno_m, config)
solver_p6.run(print_info=True, plot=True)

A análise de complexidade computacional avalia o tempo de montagem e de resolução do sistema linear para redes com diferentes níveis de refinamento. A fase de montagem tem complexidade menor, pois percorre cada aresta uma única vez. Já a resolução por LU densa, implementada com np.linalg.solve, apresenta um custo computacional elevado que cresce de forma polinomial com o número de nós. Esse alto custo da resolução fica evidente quando se aumenta o parâmetro levels, o que justifica o uso de solvers esparsos em redes maiores.

In [ ]:
analysis.complexity_analysis()

O estudo demonstrou que a linearidade do modelo hidráulico pode ser explorada de forma sistemática para reduzir drasticamente o custo computacional. Problemas com condições de contorno compostas por múltiplas vazões ou pressões prescritas em vários pontos,foram resolvidos por superposição de um pequeno número de sistemas base, em vez de se resolver o sistema completo a cada instante. Da mesma forma, variações apenas na viscosidade foram tratadas por simples escalonamento de uma única solução de referência. Essas estratégias mantêm a precisão e eliminam a dependência do número de passos temporais, tornando a abordagem especialmente vantajosa para simulações longas. Para redes de maior porte, a limitação está no solver denso utilizado, sendo recomendada a migração para resolvedores esparsos. Os parâmetros físicos adotados (diâmetro hidráulico de ~564 µm, viscosidade de 0,001 Pa·s e vazão de 1×10⁻⁷ m³/s) contextualizam a escala do problema e confirmam a aplicabilidade dos métodos a microescoamentos típicos.